# 👥 Customer Segmentation & Sales Forecasting
**Notebook:** RFM Analysis · K-Means Clustering · Facebook Prophet Forecasting  
**Stack:** Python · Scikit-learn · Prophet · Pandas · Matplotlib · Seaborn  
**Author:** Ragul Velmurugan | MS Business Analytics & AI · UT Dallas

---
### Table of Contents
1. [Setup & Data Generation](#1)
2. [Exploratory Data Analysis](#2)
3. [RFM Analysis](#3)
4. [RFM Scoring & Segmentation](#4)
5. [K-Means Clustering – Optimal K](#5)
6. [K-Means Fitting & Cluster Profiles](#6)
7. [PCA Visualisation](#7)
8. [Sales Forecasting with Prophet](#8)
9. [Category-Level Revenue Analysis](#9)
10. [Marketing Strategy Matrix](#10)
11. [Summary & Recommendations](#11)


## 1. Setup & Data Generation <a id='1'></a>

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from generate_data import generate_transactions
from rfm_analysis  import compute_rfm, score_rfm, label_segments
from clustering    import find_optimal_k, fit_kmeans, cluster_profiles
from forecasting   import prepare_prophet_df, fit_prophet

PALETTE = ['#E63946','#2A9D8F','#E9C46A','#F4A261','#264653',
           '#A8DADC','#457B9D','#1D3557','#606C38','#BC6C25']
sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams.update({'figure.dpi': 130, 'font.family': 'DejaVu Sans'})
print('Libraries loaded ✓')

In [ ]:
df = generate_transactions()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f'Transactions  : {len(df):,}')
print(f'Customers     : {df["CustomerID"].nunique():,}')
print(f'Products      : {df["Description"].nunique()}')
print(f'Categories    : {df["Category"].nunique()}')
print(f'Date Range    : {df["InvoiceDate"].min().date()} → {df["InvoiceDate"].max().date()}')
print(f'Total Revenue : ${df[df["Revenue"]>0]["Revenue"].sum():,.2f}')
df.head(5)

## 2. Exploratory Data Analysis <a id='2'></a>

In [ ]:
df[['Quantity','UnitPrice','Revenue']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Transaction EDA', fontsize=14, fontweight='bold')

# Revenue distribution
pos = df[df['Revenue'].between(0.01, df['Revenue'].quantile(0.99))]
axes[0].hist(pos['Revenue'], bins=50, color='#E63946', edgecolor='white', alpha=0.85)
axes[0].set_title('Revenue per Transaction'); axes[0].set_xlabel('Revenue ($)')

# Orders per customer
orders = df[df['Revenue']>0].groupby('CustomerID')['InvoiceNo'].nunique()
axes[1].hist(orders, bins=40, color='#2A9D8F', edgecolor='white', alpha=0.85)
axes[1].set_title('Orders per Customer'); axes[1].set_xlabel('Number of Orders')

# Revenue by country
top_countries = df[df['Revenue']>0].groupby('Country')['Revenue'].sum().nlargest(5)
axes[2].barh(top_countries.index[::-1], top_countries.values[::-1],
             color=PALETTE[:5], edgecolor='white')
axes[2].set_title('Revenue – Top 5 Countries'); axes[2].set_xlabel('Revenue ($)')

plt.tight_layout(); plt.show()

In [ ]:
# Monthly revenue trend
monthly = df[df['Revenue']>0].groupby(pd.Grouper(key='InvoiceDate', freq='ME'))['Revenue'].sum()

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(monthly.index, monthly.values, alpha=0.2, color='#2A9D8F')
ax.plot(monthly.index, monthly.values, color='#2A9D8F', linewidth=2)
ax.set_title('Monthly Revenue Trend (2022–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.tight_layout(); plt.show()

## 3. RFM Analysis <a id='3'></a>

In [ ]:
rfm = compute_rfm(df)
print(f'Customers in RFM table: {len(rfm):,}')
rfm[['Recency','Frequency','Monetary']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('RFM Raw Distributions', fontsize=14, fontweight='bold')

for ax, (col, color) in zip(axes, [
    ('Recency','#E63946'), ('Frequency','#2A9D8F'), ('Monetary','#E9C46A')
]):
    ax.hist(rfm[col], bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(rfm[col].median(), color='#264653', linestyle='--', linewidth=1.8,
               label=f'Median: {rfm[col].median():.0f}')
    ax.set_title(col, fontsize=13); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

## 4. RFM Scoring & Segmentation <a id='4'></a>

In [ ]:
rfm = score_rfm(rfm)
rfm = label_segments(rfm)

print('Score columns added: R_Score, F_Score, M_Score, RFM_Score, Segment')
rfm[['CustomerID','Recency','Frequency','Monetary','R_Score','F_Score','M_Score','RFM_Score','Segment']].head(10)

In [ ]:
seg_counts = rfm['Segment'].value_counts().reset_index()
seg_counts.columns = ['Segment','Count']
seg_counts['Pct'] = (seg_counts['Count']/seg_counts['Count'].sum()*100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('Customer Segment Distribution', fontsize=14, fontweight='bold')

bars = axes[0].barh(seg_counts['Segment'], seg_counts['Count'],
                    color=PALETTE[:len(seg_counts)], edgecolor='white')
for bar, (_, row) in zip(bars, seg_counts.iterrows()):
    axes[0].text(bar.get_width()+10, bar.get_y()+bar.get_height()/2,
                 f'{row["Count"]:,} ({row["Pct"]}%)', va='center', fontsize=9)
axes[0].set_title('Customers per Segment'); axes[0].set_xlim(0, seg_counts['Count'].max()*1.3)

seg_rev = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=True)
bars2 = axes[1].barh(seg_rev.index, seg_rev.values, color=PALETTE[:len(seg_rev)], edgecolor='white')
for bar, val in zip(bars2, seg_rev.values):
    axes[1].text(bar.get_width()+100, bar.get_y()+bar.get_height()/2,
                 f'${val:,.0f}', va='center', fontsize=9)
axes[1].set_title('Total Revenue per Segment'); axes[1].set_xlim(0, seg_rev.max()*1.3)

plt.tight_layout(); plt.show()

In [ ]:
# RFM Score heatmap
pivot_rf = rfm.pivot_table(values='Monetary', index='R_Score', columns='F_Score', aggfunc='mean').round(0)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot_rf, annot=True, fmt='.0f', cmap='RdYlGn',
            ax=ax, linewidths=0.5, cbar_kws={'label':'Avg Monetary ($)'})
ax.set_title('Average Monetary Value – Recency Score × Frequency Score', fontsize=12, fontweight='bold')
ax.set_xlabel('Frequency Score (1=Low → 5=High)')
ax.set_ylabel('Recency Score (1=Churned → 5=Active)')
plt.tight_layout(); plt.show()

## 5. K-Means – Optimal K Selection <a id='5'></a>

In [ ]:
# Prepare features
features = ['Recency','Frequency','Monetary']
X = rfm[features].copy()
X['Frequency'] = np.log1p(X['Frequency'])
X['Monetary']  = np.log1p(X['Monetary'])
X['Recency']   = np.log1p(X['Recency'])

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

k_range, inertias, silhouettes = find_optimal_k(X_scaled, k_range=range(2,9))
print('Elbow & Silhouette computed for k = 2..8')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimal K Selection', fontsize=14, fontweight='bold')

axes[0].plot(k_range, inertias, 'o-', color='#E63946', linewidth=2, markersize=7)
axes[0].axvline(4, color='#2A9D8F', linestyle='--', linewidth=1.5, label='k=4 selected')
axes[0].set_title('Elbow Method (Inertia)'); axes[0].set_xlabel('k'); axes[0].legend()

axes[1].plot(k_range, silhouettes, 's-', color='#2A9D8F', linewidth=2, markersize=7)
axes[1].axvline(4, color='#E63946', linestyle='--', linewidth=1.5, label='k=4 selected')
axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('k'); axes[1].legend()

plt.tight_layout(); plt.show()

## 6. K-Means Fitting & Cluster Profiles <a id='6'></a>

In [ ]:
rfm_clustered, scaler_fit, km_model, X_sc = fit_kmeans(rfm, n_clusters=4)
profiles = cluster_profiles(rfm_clustered)
profiles

In [ ]:
# Boxplots per cluster
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('RFM Distribution per Cluster', fontsize=14, fontweight='bold')

for ax, col in zip(axes, ['Recency','Frequency','Monetary']):
    data_list = [rfm_clustered.loc[rfm_clustered['Cluster']==c, col].values
                 for c in sorted(rfm_clustered['Cluster'].unique())]
    bp = ax.boxplot(data_list, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color); patch.set_alpha(0.75)
    ax.set_title(col); ax.set_xlabel('Cluster')
    ax.set_xticklabels([f'C{c}' for c in sorted(rfm_clustered['Cluster'].unique())])

plt.tight_layout(); plt.show()

## 7. PCA Visualisation of Clusters <a id='7'></a>

In [ ]:
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sc)
explained = pca.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('K-Means Customer Segments', fontsize=14, fontweight='bold')

ax = axes[0]
for cl in sorted(rfm_clustered['Cluster'].unique()):
    mask = rfm_clustered['Cluster'] == cl
    label_str = profiles.loc[profiles['Cluster']==cl,'Cluster_Label'].values
    label_str = label_str[0] if len(label_str) else f'Cluster {cl}'
    ax.scatter(X_pca[mask,0], X_pca[mask,1], s=12, alpha=0.4,
               color=PALETTE[cl], label=f'C{cl}: {label_str}')
ax.set_title(f'PCA (PC1={explained[0]:.1f}%, PC2={explained[1]:.1f}%)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend(fontsize=8, markerscale=3)

ax = axes[1]
cr = rfm_clustered.groupby('Cluster')[['Recency','Frequency','Monetary']].mean().reset_index()
x  = np.arange(len(cr)); w = 0.25
ax.bar(x-w,  cr['Recency'],       w, label='Recency',      color='#E63946', alpha=0.85)
ax.bar(x,    cr['Frequency'],     w, label='Frequency',    color='#2A9D8F', alpha=0.85)
ax.bar(x+w,  cr['Monetary']/100, w, label='Monetary/100', color='#E9C46A', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([f'Cluster {c}' for c in cr['Cluster']])
ax.set_title('Mean RFM per Cluster'); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

## 8. Sales Forecasting with Facebook Prophet <a id='8'></a>

In [ ]:
ts = prepare_prophet_df(df, freq='W')
print(f'Weekly time series: {len(ts)} data points')
ts.head()

In [ ]:
model, forecast, train, test, metrics = fit_prophet(ts, periods=26, freq='W')
print('\nForecast Evaluation Metrics (holdout test set):')
print(f"  MAPE : {metrics['MAPE']}%")
print(f"  MAE  : ${metrics['MAE']:,.0f}")
print(f"  RMSE : ${metrics['RMSE']:,.0f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Weekly Revenue Forecast – Facebook Prophet', fontsize=14, fontweight='bold')

ax = axes[0]
ax.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'],
                alpha=0.2, color='#2A9D8F', label='CI')
ax.plot(forecast['ds'], forecast['yhat'], color='#2A9D8F', linewidth=2, label='Forecast')
ax.scatter(train['ds'], train['y'], s=8, color='#264653', alpha=0.6, label='Actual (train)')
ax.scatter(test['ds'],  test['y'],  s=12, color='#E63946', alpha=0.85, label='Actual (test)')
ax.axvline(train['ds'].max(), color='gray', linestyle='--', linewidth=1.2, label='Split')
ax.set_title('Revenue Forecast with Confidence Bands')
ax.set_xlabel('Date'); ax.set_ylabel('Weekly Revenue ($)'); ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))

ax = axes[1]
comps = model.predict(train)
ax.plot(comps['ds'], comps['trend'], color='#E63946', linewidth=2, label='Trend')
ax.fill_between(comps['ds'], comps['trend']*0.95, comps['trend']*1.05, alpha=0.15, color='#E63946')
ax.set_title('Underlying Revenue Trend')
ax.set_xlabel('Date'); ax.set_ylabel('Trend ($)'); ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.tight_layout(); plt.show()

## 9. Category-Level Revenue Analysis <a id='9'></a>

In [ ]:
monthly_cat = (
    df[df['Revenue']>0]
    .groupby([pd.Grouper(key='InvoiceDate',freq='ME'),'Category'])['Revenue']
    .sum().reset_index()
)
top_cats = df.groupby('Category')['Revenue'].sum().nlargest(5).index.tolist()

fig, axes = plt.subplots(2,1, figsize=(16,10))
fig.suptitle('Category Revenue Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
for i,cat in enumerate(top_cats):
    sub = monthly_cat[monthly_cat['Category']==cat]
    ax.plot(sub['InvoiceDate'], sub['Revenue'], marker='o', markersize=4,
            linewidth=2, color=PALETTE[i], label=cat)
ax.set_title('Monthly Revenue – Top 5 Categories')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(fontsize=9)

ax = axes[1]
cat_total = df[df['Revenue']>0].groupby('Category')['Revenue'].sum().sort_values(ascending=False)
bars = ax.bar(cat_total.index, cat_total.values, color=PALETTE[:len(cat_total)], edgecolor='white')
ax.bar_label(bars, fmt='$%.0f', rotation=45, padding=4, fontsize=8)
ax.set_title('Total Revenue by Category (Full Period)')
ax.tick_params(axis='x', rotation=30)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.tight_layout(); plt.show()

## 10. Marketing Strategy Matrix <a id='10'></a>

In [ ]:
seg_stats = rfm.groupby('Segment').agg(
    avg_recency  = ('Recency','mean'),
    avg_monetary = ('Monetary','mean'),
    count        = ('CustomerID','count')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(seg_stats['avg_recency'], seg_stats['avg_monetary'],
           s=seg_stats['count']*0.8, c=range(len(seg_stats)),
           cmap='tab10', alpha=0.8, edgecolors='white', linewidth=1.5)
for _, row in seg_stats.iterrows():
    ax.annotate(row['Segment'], (row['avg_recency'], row['avg_monetary']),
                xytext=(6,4), textcoords='offset points', fontsize=9, fontweight='bold')
ax.axhline(seg_stats['avg_monetary'].median(), color='gray', linestyle='--', alpha=0.5)
ax.axvline(seg_stats['avg_recency'].median(), color='gray', linestyle='--', alpha=0.5)
ax.set_title('Customer Segment – Recency vs Value Matrix\n(bubble size = # customers)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Avg Recency (days)  ← Better'); ax.set_ylabel('Avg Monetary Value ($)')
ax.invert_xaxis()
plt.tight_layout(); plt.show()

## 11. Summary & Marketing Recommendations <a id='11'></a>

### Model Performance
| Metric | Value |
|---|---|
| K-Means Silhouette Score | ~0.27 |
| Prophet MAPE (holdout) | ~9% |
| Prophet MAE | ~$600/week |

### Marketing Recommendations by Segment

| Segment | Priority | Strategy |
|---|---|---|
| **Champions** | Retain | Loyalty programme, upsell premium, early access |
| **Loyal Customers** | Grow | Cross-sell, referral rewards |
| **At Risk** | Win-back | Personalised email + exclusive discount |
| **New Customers** | Nurture | Onboarding sequence, first-order incentive |
| **Lost** | Reactivate | Heavy discount + re-engagement campaign |
| **Hibernating** | Remind | Time-limited offers, value reminder |

### Cluster-Based Targeting
| Cluster | Profile | Action |
|---|---|---|
| High-Value Active | Champions | VIP programme |
| High-Value Inactive | At Risk | Urgent win-back |
| Low-Value Active | New/Promising | Spend encouragement |
| Low-Value Inactive | Lost | Budget reactivation |

---
*Generated with Python · scikit-learn · Facebook Prophet | Ragul Velmurugan · UT Dallas*
